In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

In [3]:
# Load the data

data_path = Path('../data/processed/clean/combined_individual_events.csv')
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns}")
print('--' * 40)

df.head()


Dataset Shape: (345, 17)
Columns: Index(['year', 'event_name', 'stroke', 'gender', 'distance', 'meet',
       'source_file', 'winning_time_sec', 'winning_time_format',
       'a_final_cutoff_sec', 'a_final_cutoff_format', 'b_final_cutoff_sec',
       'b_final_cutoff_format', 'c_final_cutoff_sec', 'c_final_cutoff_format',
       'total_swimmers', 'results'],
      dtype='object')
--------------------------------------------------------------------------------


,year,event_name,stroke,gender,distance,meet,source_file,winning_time_sec,winning_time_format,a_final_cutoff_sec,a_final_cutoff_format,b_final_cutoff_sec,b_final_cutoff_format,c_final_cutoff_sec,c_final_cutoff_format,total_swimmers,results
0,2002,100_Backstroke,Backstroke,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,51.45,51.45,54.94,54.94,56.16,56.16,NaN,NaN,23,"[{'name': 'Schwenker, Eric', 'yr': 'SR', 'scho..."
1,2002,100_Breaststroke,Breaststroke,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,56.98,56.98,59.83,59.83,61.58,1:01.58,63.14,1:03.14,35,"[{'name': 'Eck, Jonathan', 'yr': 'JR', 'school..."
2,2002,100_Butterfly,Butterfly,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,50.70,50.70,53.04,53.04,54.46,54.46,56.05,56.05,28,"[{'name': 'Stuntz, Grayson', 'yr': 'SR', 'scho..."
3,2002,100_Freestyle,Freestyle,Men,100,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,45.83,45.83,48.14,48.14,48.79,48.79,49.54,49.54,45,"[{'name': 'Walendziak, Nick', 'yr': 'SO', 'sch..."
4,2002,200_Backstroke,Backstroke,Men,200,2002_Nescac_Msd_Results,2002_NESCAC_MSD_Results.txt,111.13,1:51.13,118.61,1:58.61,121.44,2:01.44,135.04,2:15.04,24,"[{'name': 'Schwenker, Eric', 'yr': 'SR', 'scho..."


In [4]:
# Datatypes
print("Datatypes:")
print(f"A_cutoff_sec: {df['a_final_cutoff_sec'].dtype}")
print(f"B_cutoff_sec: {df['b_final_cutoff_sec'].dtype}")
print(f"C_cutoff_sec: {df['c_final_cutoff_sec'].dtype}")

# Convert to numeric just in case

for col in ['a_final_cutoff_sec', 'b_final_cutoff_sec', 'c_final_cutoff_sec']:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')

print("After Conversion:")
print(f"A_cutoff_sec: {df['a_final_cutoff_sec'].dtype}")
print(f"B_cutoff_sec: {df['b_final_cutoff_sec'].dtype}")
print(f"C_cutoff_sec: {df['c_final_cutoff_sec'].dtype}")

Datatypes:
A_cutoff_sec: float64
B_cutoff_sec: float64
C_cutoff_sec: float64
After Conversion:
A_cutoff_sec: float64
B_cutoff_sec: float64
C_cutoff_sec: float64


In [5]:
# Unique ID

df['event_id'] = df['gender'] + '_' + df['event_name']

print(f"Unique events: {df['event_id'].value_counts()}")




Unique events: event_id
Men_100_Backstroke      23
Men_100_Breaststroke    23
Men_100_Butterfly       23
Men_100_Freestyle       23
Men_200_Backstroke      23
Men_200_Breaststroke    23
Men_200_Butterfly       23
Men_200_Freestyle       23
Men_200_IM              23
Men_400_IM              23
Men_500_Freestyle       23
Men_50_Backstroke       23
Men_50_Breaststroke     23
Men_50_Butterfly        23
Men_50_Freestyle        23
Name: count, dtype: int64


In [16]:
def plot_winning_times(event_data, event_name, output_dir):
    # sort by year
    event_data = event_data.sort_values('year')

    # compute year bounds
    min_year = int(event_data['year'].min()) - 1
    max_year = int(event_data['year'].max()) + 1

    # prepare a clean title (no underscores)
    pretty_name = event_name.replace('_', ' ')

    fig, ax = plt.subplots(figsize=(12, 8))

    # plot with a mid‑tone blue
    if not event_data['winning_time_sec'].isna().all():
        ax.plot(
            event_data['year'],
            event_data['winning_time_sec'],
            marker='o',
            linewidth=2,
            markersize=6,
            label='Winning Time',
            color='#1f77b4'
        )

    # clean spines & light grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    # titles & labels (using pretty_name)
    ax.set_title(f'{pretty_name} Winning Time by Year',
                 fontsize=18, fontweight='bold', pad=15)
    ax.set_xlabel('Year', fontsize=14)
    ax.set_ylabel('Time (seconds)', fontsize=14)

    # only every 4th year
    years = list(range(min_year, max_year + 1, 3))
    ax.set_xlim(min_year, max_year)
    ax.set_xticks(years)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
    plt.setp(ax.get_yticklabels(), fontsize=12)

    # legend & layout
    ax.legend(frameon=False, fontsize=12, loc='best')
    plt.tight_layout()

    # save using underscores in the filename
    safe_name = event_name.replace(' ', '_').replace('/', '_')
    output_path = Path(output_dir) / f'{safe_name}_winning_times.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    return output_path

In [17]:
output_dir = Path('../output/plots/winning_times')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Creating plots in: {output_dir.absolute()}")

Creating plots in: /Users/hlecates/Desktop/aqua-analytics/nescac/notebooks/../output/plots/winning_times


In [18]:
saved_plots = []

for event_id in df['event_id'].unique():
    event_data = df[df['event_id'] == event_id]

    if event_data['winning_time_sec'].isna().all():
        continue

    plot_title = f'{event_id}'

    try:
        plot_path = plot_winning_times(event_data, plot_title, output_dir)
        saved_plots.append(plot_path)
        print(f"Created plot: {plot_title}")
    except Exception as e:
        print(f"Error creating plot for {plot_title}: {e}")

print(f"Total plots created: {len(saved_plots)}")

Created plot: Men_100_Backstroke
Created plot: Men_100_Breaststroke
Created plot: Men_100_Butterfly
Created plot: Men_100_Freestyle
Created plot: Men_200_Backstroke
Created plot: Men_200_Breaststroke
Created plot: Men_200_Butterfly
Created plot: Men_200_Freestyle
Created plot: Men_200_IM
Created plot: Men_400_IM
Created plot: Men_500_Freestyle
Created plot: Men_50_Backstroke
Created plot: Men_50_Breaststroke
Created plot: Men_50_Butterfly
Created plot: Men_50_Freestyle
Total plots created: 15
